# pyplotrs — core plots verify pass

Each core plot type is rendered **twice**: first with **pyplotrs** (imported as
`pp`), then the same data with **matplotlib** (`plt`) directly below it as a
reference for visual checking. Use **Run All**, then compare each pyplotrs plot
against its matplotlib version.

pyplotrs cells end with a bare `fig`, which displays via `Figure._repr_png_`
(a bare figure shows inline). matplotlib cells use the `%matplotlib inline`
backend and end with `plt.show()`. Both plot **identical data** (stdlib
`random`, seeded — no numpy), and figure sizes match: pyplotrs `figsize` is in
**points**, so the matplotlib references convert with `pt2in` (points → inches;
72 pt = 1 in). The matplotlib cells never touch the RNG (they reuse variables
from the pyplotrs cell above), so the two libraries always see the same numbers.

**Part 1 - core set:** line · bar · box · pie · scatter · errorbar · violin ·
polar · axhspan/axvspan.  **Also kept on `dev`:** histogram · fill_between ·
imshow.  **Part 2** then sweeps the rest of pyplotrs' plotting API - the basic
family, binned/statistical, contours, 2D arrays, vector fields, spans, shapes,
text, axes composition, and the 3D projection - and lists, at its head, the
matplotlib plot types pyplotrs does *not* have.

For each plot, judge pyplotrs against its philosophy — sensible defaults, clean
spacing, colourblind-safe palette, crisp typography, publication-ready with zero


In [ ]:
import inspect
import math, random
import pyplotrs as pp
import matplotlib.pyplot as plt
%matplotlib inline

# Render matplotlib at pyplotrs' density (150 dpi) so the two show at the
# same on-screen size for side-by-side comparison.
plt.rcParams["figure.dpi"] = 150


def pt2in(w, h):
    """pyplotrs figsize is in points; matplotlib wants inches (72 pt = 1 in)."""
    return (w / 72, h / 72)


# Read pyplotrs' default figure size rather than hardcoding it: the two halves
# of every comparison below have to be the same size, and a default that drifts
# would quietly turn this notebook into a size comparison.
DEFAULT_FIGSIZE = inspect.signature(pp.subplots).parameters["figsize"].default


random.seed(0)
xs = [i * 0.25 for i in range(40)]
dists = [[random.gauss(mu, sd) for _ in range(80)]
         for mu, sd in [(0.0, 1.0), (1.8, 1.3), (-1.2, 0.8), (2.5, 1.6)]]
labels4 = ["alpha", "beta", "gamma", "delta"]
pos4 = [1, 2, 3, 4]
print("pyplotrs ready - body font:", pp.resolved_font_name())

## 1 · Line
Multiple series, dashed style, legend, axis labels.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs], label="sin")
ax.line(xs, [math.sin(x) * math.exp(-0.15 * x) for x in xs],
        label="damped", linestyle="dashed")
ax.set(title="Line", xlabel="t", ylabel="amplitude")
ax.legend()
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs], label="sin")
ax.plot(xs, [math.sin(x) * math.exp(-0.15 * x) for x in xs],
        label="damped", linestyle="dashed")
ax.set(title="Line (matplotlib)", xlabel="t", ylabel="amplitude")
ax.legend()
plt.show()

## 2 · Bar
Categorical bars with named ticks.

In [ ]:
fig, ax = pp.subplots()
ax.bar(pos4, [5.0, 3.0, 7.5, 4.2])
ax.set(title="Bar", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.bar(pos4, [5.0, 3.0, 7.5, 4.2])
ax.set(title="Bar (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 3 · Box
Box-and-whisker over four distributions.

In [ ]:
fig, ax = pp.subplots()
ax.boxplot(dists, positions=pos4)
ax.set(title="Box", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.boxplot(dists, positions=pos4)
ax.set(title="Box (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 4 · Pie
Proportions with labels; equal aspect, frame off.

In [ ]:
fig, ax = pp.subplots()
ax.pie([35, 25, 22, 18], labels=labels4)
ax.set(title="Pie")
fig

In [ ]:
# matplotlib reference (startangle=90 to match pyplotrs' default)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.pie([35, 25, 22, 18], labels=labels4, startangle=90)
ax.set(title="Pie (matplotlib)")
plt.show()

## 5 · Scatter (color-mapped)
Per-point color by a third value (`c=`), with a colorbar.

In [ ]:
random.seed(1)
n = 200
sx = [random.gauss(0.0, 1.0) for _ in range(n)]
sy = [random.gauss(0.0, 1.0) for _ in range(n)]
cv = [math.hypot(x, y) for x, y in zip(sx, sy)]
fig, ax = pp.subplots()
sc = ax.scatter(sx, sy, c=cv, size=24)
fig.colorbar(sc, label="radius")
ax.set(title="Scatter (color-mapped)", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses sx, sy, cv; note size= -> s=)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
sc = ax.scatter(sx, sy, c=cv, s=24)
fig.colorbar(sc, ax=ax, label="radius")
ax.set(title="Scatter (matplotlib)", xlabel="x", ylabel="y")
plt.show()

## 6 · Errorbar
Symmetric x and y error bars with caps.

In [ ]:
ex = list(range(1, 8))
ey = [1.0, 2.1, 1.7, 3.2, 2.8, 3.9, 3.5]
fig, ax = pp.subplots()
ax.errorbar(ex, ey, yerr=[0.3] * len(ex), xerr=[0.15] * len(ex))
ax.set(title="Errorbar", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (marker/capsize match pyplotrs' defaults)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.errorbar(ex, ey, yerr=[0.3] * len(ex), xerr=[0.15] * len(ex),
            marker="o", capsize=3)
ax.set(title="Errorbar (matplotlib)", xlabel="x", ylabel="y")
plt.show()

## 7 · Violin
KDE violins over the same four distributions.

In [ ]:
fig, ax = pp.subplots()
ax.violinplot(dists, positions=pos4)
ax.set(title="Violin", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.violinplot(dists, positions=pos4)
ax.set(title="Violin (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 8 · Polar
Line on a polar projection (radians, CCW from East), with a legend.

In [ ]:
theta = [i * math.pi / 180 for i in range(361)]
fig, ax = pp.subplots(projection="polar", figsize=(360, 360))
ax.plot(theta, [abs(math.cos(2 * t)) for t in theta], label="rose")
ax.plot(theta, [t / (2 * math.pi) for t in theta], label="spiral", linestyle="dashed")
ax.set(title="Polar")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses theta)
fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=pt2in(360, 360))
ax.plot(theta, [abs(math.cos(2 * t)) for t in theta], label="rose")
ax.plot(theta, [t / (2 * math.pi) for t in theta], label="spiral", linestyle="dashed")
ax.set(title="Polar (matplotlib)")
ax.legend()
plt.show()

## 9 · axhspan / axvspan
Shaded bands + reference lines over a line.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs])
ax.axhspan(0.5, 1.0, color="C1", alpha=0.15)
ax.axvspan(2.0, 4.0, color="C2", alpha=0.15)
ax.axhline(0.0)
ax.axvline(5.0, linestyle="dashed")
ax.set(title="axhspan / axvspan + reference lines", xlabel="t", ylabel="sin")
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs])
ax.axhspan(0.5, 1.0, color="C1", alpha=0.15)
ax.axvspan(2.0, 4.0, color="C2", alpha=0.15)
ax.axhline(0.0)
ax.axvline(5.0, linestyle="dashed")
ax.set(title="axhspan / axvspan (matplotlib)", xlabel="t", ylabel="sin")
plt.show()

## Also kept · Histogram

In [ ]:
hsample = [random.gauss(0.0, 1.0) for _ in range(1000)]
fig, ax = pp.subplots()
ax.hist(hsample, bins=24)
ax.set(title="Histogram", xlabel="value", ylabel="count")
fig

In [ ]:
# matplotlib reference (reuses hsample)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.hist(hsample, bins=24)
ax.set(title="Histogram (matplotlib)", xlabel="value", ylabel="count")
plt.show()

## Also kept · fill_between

In [ ]:
base = [math.sin(x) for x in xs]
lo = [v - 0.3 for v in base]
hi = [v + 0.3 for v in base]
fig, ax = pp.subplots()
ax.fill_between(xs, lo, hi, label="band")
ax.line(xs, base, label="mean")
ax.set(title="fill_between", xlabel="t", ylabel="y")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses base, lo, hi)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.fill_between(xs, lo, hi, label="band")
ax.plot(xs, base, label="mean")
ax.set(title="fill_between (matplotlib)", xlabel="t", ylabel="y")
ax.legend()
plt.show()

## Also kept · imshow (+ colorbar)

In [ ]:
grid = [[math.sin(0.3 * i) * math.cos(0.3 * j) for j in range(30)] for i in range(20)]
fig, ax = pp.subplots()
im = ax.imshow(grid, cmap="viridis")
fig.colorbar(im, label="value")
ax.set(title="imshow")
fig

In [ ]:
# matplotlib reference (reuses grid)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
im = ax.imshow(grid, cmap="viridis")
fig.colorbar(im, ax=ax, label="value")
ax.set(title="imshow (matplotlib)")
plt.show()

## Part 2 · the rest of the plotting API

Part 1 above is the hand-checked core set. Everything below is the rest of
pyplotrs' plotting surface, grouped the way matplotlib's `Axes` API groups it,
and rendered against a matplotlib reference in exactly the same way.

**Not in pyplotrs** (so deliberately absent below), for the record:
the spectral family (`acorr`, `xcorr`, `psd`, `csd`, `cohere`, `specgram`,
`magnitude_spectrum`, `angle_spectrum`, `phase_spectrum`), the unstructured
triangular grid family (`tripcolor`, `triplot`, `tricontour`, `tricontourf`),
`barbs`, `ecdf`, `bxp`, `bar_label`, `clabel`, `quiverkey`, `pcolorfast`,
`table`, `indicate_inset`, and 3D `stem`.

Note the naming differences from matplotlib: `plot` → `line`, and the patch
classes (`Rectangle`, `Circle`, `Ellipse`, `Polygon`) are axes methods
(`rectangle`, `circle`, `ellipse`, `polygon`) rather than `add_patch` arguments.

In [ ]:
# Shared data for the sections below (stdlib only, seeded - matplotlib reuses it).
random.seed(2)

# Rectilinear grid + a smooth scalar field on it, for contour / pcolormesh /
# surface / wireframe.
gx = [-3.0 + 6.0 * i / 29 for i in range(30)]
gy = [-2.0 + 4.0 * j / 19 for j in range(20)]
Z = [[math.exp(-(x * x + y * y) / 4) * math.sin(1.5 * x) * math.cos(1.2 * y)
      for x in gx] for y in gy]

# Coarser grid + a vector field on it, for quiver / streamplot.
vx = [-2.0 + 4.0 * i / 11 for i in range(12)]
vy = [-2.0 + 4.0 * j / 11 for j in range(12)]
U = [[-y for x in vx] for y in vy]
V = [[x for x in vx] for y in vy]

# A big correlated point cloud, for hist2d / hexbin.
n2 = 4000
bx, by = [], []
for _ in range(n2):
    a = random.gauss(0.0, 1.0)
    b = random.gauss(0.0, 1.0)
    bx.append(a)
    by.append(0.7 * a + 0.7 * b)

print("shared data ready")

### barh
Horizontal bars - `barh(y, width)`.

In [ ]:
fig, ax = pp.subplots()
ax.barh(pos4, [3.2, 5.1, 2.4, 4.6], label="score")
ax.set(title="barh", xlabel="score", ylabel="group",
       yticks=pos4, yticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.barh(pos4, [3.2, 5.1, 2.4, 4.6], label="score")
ax.set(title="barh (matplotlib)", xlabel="score", ylabel="group",
       yticks=pos4, yticklabels=labels4)
plt.show()

### step / stairs
`step(where=)` for a sampled signal; `stairs(values, edges)` for
bin-edge data (a histogram outline).

In [ ]:
sx6 = list(range(10))
sy6 = [random.gauss(0.0, 1.0) for _ in range(10)]
edges = [i * 0.5 for i in range(11)]
vals = [abs(random.gauss(0.0, 1.0)) for _ in range(10)]

fig, (ax1, ax2) = pp.subplots(1, 2, figsize=(520, 200))
ax1.step(sx6, sy6, where="mid", label="mid")
ax1.set(title="step", xlabel="i", ylabel="value")
ax1.legend()
ax2.stairs(vals, edges, fill=True)
ax2.set(title="stairs", xlabel="edge", ylabel="count")
fig

In [ ]:
# matplotlib reference (reuses sx6, sy6, edges, vals)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=pt2in(520, 200))
ax1.step(sx6, sy6, where="mid", label="mid")
ax1.set(title="step (matplotlib)", xlabel="i", ylabel="value")
ax1.legend()
ax2.stairs(vals, edges, fill=True)
ax2.set(title="stairs (matplotlib)", xlabel="edge", ylabel="count")
plt.show()

### stem
Stems from a baseline with a marker head.

In [ ]:
tx = [i * 0.4 for i in range(24)]
ty = [math.exp(-0.12 * t) * math.cos(2.2 * t) for t in tx]
fig, ax = pp.subplots()
ax.stem(tx, ty)
ax.set(title="stem", xlabel="t", ylabel="amplitude")
fig

In [ ]:
# matplotlib reference (reuses tx, ty)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.stem(tx, ty)
ax.set(title="stem (matplotlib)", xlabel="t", ylabel="amplitude")
plt.show()

### stackplot
Stacked filled areas - part-to-whole over a shared x.

In [ ]:
sk = list(range(12))
s1 = [2 + math.sin(i * 0.6) for i in sk]
s2 = [3 + math.cos(i * 0.4) for i in sk]
s3 = [1.5 + 0.4 * i ** 0.5 for i in sk]
fig, ax = pp.subplots()
ax.stackplot(sk, s1, s2, s3, labels=["alpha", "beta", "gamma"])
ax.set(title="stackplot", xlabel="month", ylabel="volume")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses sk, s1, s2, s3)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.stackplot(sk, s1, s2, s3, labels=["alpha", "beta", "gamma"])
ax.set(title="stackplot (matplotlib)", xlabel="month", ylabel="volume")
ax.legend()
plt.show()

### broken_barh
Runs of intervals on one row - a Gantt / uptime strip.

In [ ]:
fig, ax = pp.subplots()
ax.broken_barh([(0, 3), (5, 2.5), (9, 4)], (10, 6), label="host A")
ax.broken_barh([(1.5, 2), (6, 4.5)], (20, 6), label="host B")
ax.set(title="broken_barh", xlabel="hour", ylabel="host",
       yticks=[13, 23], yticklabels=["A", "B"], xlim=(0, 14))
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.broken_barh([(0, 3), (5, 2.5), (9, 4)], (10, 6), label="host A")
ax.broken_barh([(1.5, 2), (6, 4.5)], (20, 6), label="host B")
ax.set(title="broken_barh (matplotlib)", xlabel="hour", ylabel="host",
       yticks=[13, 23], yticklabels=["A", "B"], xlim=(0, 14))
plt.show()

### eventplot
Raster of event times - one row per series.

**API difference:** pyplotrs reads `lineoffsets` as the *spacing between*
the rows of a single call (row `i` sits at `lineoffsets * i`), so all rows
go in one call. matplotlib reads it as each row's *absolute* offset, so the
reference passes the explicit sequence `0..5` to land on the same rows.

In [ ]:
trains = [sorted(random.uniform(0, 10) for _ in range(random.randint(12, 28)))
          for _ in range(6)]
fig, ax = pp.subplots()
ax.eventplot(trains, lineoffsets=1, linelengths=0.7)
ax.set(title="eventplot", xlabel="time (s)", ylabel="trial",
       yticks=list(range(6)))
fig

In [ ]:
# matplotlib reference (reuses trains)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.eventplot(trains, lineoffsets=list(range(6)), linelengths=0.7)
ax.set(title="eventplot (matplotlib)", xlabel="time (s)", ylabel="trial",
       yticks=list(range(6)))
plt.show()

### fill_betweenx
The transpose of `fill_between` - a band about a vertical profile.

In [ ]:
py_ = [i * 0.3 for i in range(30)]
px = [math.sin(y * 0.8) for y in py_]
plo = [v - 0.25 for v in px]
phi = [v + 0.25 for v in px]
fig, ax = pp.subplots()
ax.fill_betweenx(py_, plo, phi, label="band")
ax.line(px, py_, label="profile")
ax.set(title="fill_betweenx", xlabel="value", ylabel="depth")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses py_, px, plo, phi)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.fill_betweenx(py_, plo, phi, alpha=0.3, label="band")
ax.plot(px, py_, label="profile")
ax.set(title="fill_betweenx (matplotlib)", xlabel="value", ylabel="depth")
ax.legend()
plt.show()

### hlines / vlines
Bounded horizontal and vertical rules (`axhline`/`axvline` below span
the whole axes instead).

In [ ]:
fig, ax = pp.subplots()
ax.hlines([1, 2, 3], 0.5, 4.5, linestyle="dashed", label="hlines")
ax.vlines([1, 2, 3, 4], 0.5, 3.5, label="vlines")
ax.set(title="hlines / vlines", xlabel="x", ylabel="y",
       xlim=(0, 5), ylim=(0, 4))
ax.legend()
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.hlines([1, 2, 3], 0.5, 4.5, linestyle="dashed", label="hlines")
ax.vlines([1, 2, 3, 4], 0.5, 3.5, label="vlines")
ax.set(title="hlines / vlines (matplotlib)", xlabel="x", ylabel="y",
       xlim=(0, 5), ylim=(0, 4))
ax.legend()
plt.show()

### semilogx / semilogy / loglog
Log-scaled convenience wrappers around `line`.

In [ ]:
lx = [10 ** (i / 8) for i in range(25)]
ly = [v ** 1.6 for v in lx]
fig, (a1, a2, a3) = pp.subplots(1, 3, figsize=(700, 200))
a1.semilogx(lx, [math.log(v) for v in lx])
a1.set(title="semilogx", xlabel="x", ylabel="log x")
a2.semilogy(range(25), [10 ** (i / 6) for i in range(25)])
a2.set(title="semilogy", xlabel="i", ylabel="y")
a3.loglog(lx, ly)
a3.set(title="loglog", xlabel="x", ylabel="x^1.6")
fig

In [ ]:
# matplotlib reference (reuses lx, ly)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=pt2in(700, 200))
a1.semilogx(lx, [math.log(v) for v in lx])
a1.set(title="semilogx (mpl)", xlabel="x", ylabel="log x")
a2.semilogy(range(25), [10 ** (i / 6) for i in range(25)])
a2.set(title="semilogy (mpl)", xlabel="i", ylabel="y")
a3.loglog(lx, ly)
a3.set(title="loglog (mpl)", xlabel="x", ylabel="x^1.6")
fig.tight_layout()
plt.show()

### hist2d
2D histogram of a point cloud, with a colorbar.

In [ ]:
fig, ax = pp.subplots()
h = ax.hist2d(bx, by, bins=30)
fig.colorbar(h, label="count")
ax.set(title="hist2d", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses bx, by)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
h = ax.hist2d(bx, by, bins=30)
fig.colorbar(h[3], ax=ax, label="count")
ax.set(title="hist2d (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### hexbin
Hexagonal binning - the same cloud on a hex lattice.

In [ ]:
fig, ax = pp.subplots()
hb = ax.hexbin(bx, by, gridsize=24)
fig.colorbar(hb, label="count")
ax.set(title="hexbin", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses bx, by)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
hb = ax.hexbin(bx, by, gridsize=24)
fig.colorbar(hb, ax=ax, label="count")
ax.set(title="hexbin (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### contour
Line contours of the shared field `Z`.

In [ ]:
fig, ax = pp.subplots()
ax.contour(gx, gy, Z, levels=10)
ax.set(title="contour", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses gx, gy, Z)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.contour(gx, gy, Z, levels=10)
ax.set(title="contour (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### contourf
Filled contours of the same field, with a colorbar.

In [ ]:
fig, ax = pp.subplots()
cf = ax.contourf(gx, gy, Z, levels=12)
fig.colorbar(cf, label="z")
ax.set(title="contourf", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses gx, gy, Z)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
cf = ax.contourf(gx, gy, Z, levels=12)
fig.colorbar(cf, ax=ax, label="z")
ax.set(title="contourf (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### pcolormesh / pcolor
The field drawn as cells on its coordinate grid, rather than resampled to
pixels like `imshow`.

In [ ]:
fig, (a1, a2) = pp.subplots(1, 2, figsize=(560, 200))
m1 = a1.pcolormesh(gx, gy, Z)
fig.colorbar(m1, label="z")
a1.set(title="pcolormesh", xlabel="x", ylabel="y")
m2 = a2.pcolor(gx, gy, Z, cmap="magma")
fig.colorbar(m2, label="z")
a2.set(title="pcolor", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses gx, gy, Z)
fig, (a1, a2) = plt.subplots(1, 2, figsize=pt2in(560, 200))
m1 = a1.pcolormesh(gx, gy, Z)
fig.colorbar(m1, ax=a1)
a1.set(title="pcolormesh (mpl)", xlabel="x", ylabel="y")
m2 = a2.pcolor(gx, gy, Z, cmap="magma")
fig.colorbar(m2, ax=a2)
a2.set(title="pcolor (mpl)", xlabel="x", ylabel="y")
fig.tight_layout()
plt.show()

### matshow / spy
Matrix conventions: `matshow` color-maps every entry, `spy` marks the
non-zeros of a sparse matrix.

In [ ]:
mat = [[(i * j) % 7 for j in range(12)] for i in range(12)]
sparse = [[(1 if (i * j) % 5 == 0 else 0) for j in range(20)] for i in range(20)]
fig, (a1, a2) = pp.subplots(1, 2, figsize=(520, 220))
ms = a1.matshow(mat)
fig.colorbar(ms, label="value")
a1.set(title="matshow")
a2.spy(sparse)
a2.set(title="spy")
fig

In [ ]:
# matplotlib reference (reuses mat, sparse)
fig, (a1, a2) = plt.subplots(1, 2, figsize=pt2in(520, 220))
ms = a1.matshow(mat)
fig.colorbar(ms, ax=a1)
a1.set(title="matshow (mpl)")
a2.spy(sparse)
a2.set(title="spy (mpl)")
fig.tight_layout()
plt.show()

### quiver
Arrow field - the rotational field `(-y, x)` on the shared coarse grid.

In [ ]:
fig, ax = pp.subplots()
QX = [[x for x in vx] for y in vy]   # quiver wants x/y meshed to match u/v
QY = [[y for x in vx] for y in vy]
ax.quiver(QX, QY, U, V, scale=0.35)
ax.set(title="quiver", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses vx, vy, U, V)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.quiver(vx, vy, U, V)
ax.set(title="quiver (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### streamplot
Streamlines traced through the same field.

In [ ]:
fig, ax = pp.subplots()
ax.streamplot(vx, vy, U, V, density=1.2)
ax.set(title="streamplot", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses vx, vy, U, V). matplotlib's streamplot
# requires ndarrays, so numpy is used purely to reshape - the numbers are
# still the stdlib-generated ones pyplotrs saw.
import numpy as np

fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.streamplot(np.array(vx), np.array(vy), np.array(U), np.array(V), density=1.2)
ax.set(title="streamplot (matplotlib)", xlabel="x", ylabel="y")
plt.show()

### axhline / axvline / axline
Full-width reference rules; `axline` takes a point plus a slope.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs])
ax.axhline(0.0, linestyle="dashed")
ax.axvline(5.0, linestyle="dotted")
ax.axline((0, -1), slope=0.15)
ax.set(title="axhline / axvline / axline", xlabel="t", ylabel="amplitude")
fig

In [ ]:
# matplotlib reference (reuses xs)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs])
ax.axhline(0.0, linestyle="dashed")
ax.axvline(5.0, linestyle="dotted")
ax.axline((0, -1), slope=0.15)
ax.set(title="axhline / axvline / axline (mpl)", xlabel="t", ylabel="amplitude")
plt.show()

### rectangle / circle / ellipse / polygon
pyplotrs exposes shapes as axes methods; matplotlib uses `add_patch` with a
patch class.

In [ ]:
fig, ax = pp.subplots()
ax.rectangle((0.5, 0.5), 2.0, 1.0, facecolor="#4C72B0", alpha=0.6)
ax.circle((4.0, 1.0), 0.8, facecolor="#DD8452", alpha=0.6)
ax.ellipse((1.5, 3.0), 2.0, 1.0, angle=25, facecolor="#55A868", alpha=0.6)
ax.polygon([(3.2, 2.4), (4.6, 2.4), (4.9, 3.6), (3.8, 4.2), (2.9, 3.4)],
           facecolor="#C44E52", alpha=0.6)
ax.set(title="shapes", xlabel="x", ylabel="y", xlim=(0, 6), ylim=(0, 5))
fig

In [ ]:
# matplotlib reference (patch classes rather than axes methods)
from matplotlib.patches import Circle, Ellipse, Polygon, Rectangle

fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.add_patch(Rectangle((0.5, 0.5), 2.0, 1.0, facecolor="#4C72B0", alpha=0.6))
ax.add_patch(Circle((4.0, 1.0), 0.8, facecolor="#DD8452", alpha=0.6))
ax.add_patch(Ellipse((1.5, 3.0), 2.0, 1.0, angle=25, facecolor="#55A868", alpha=0.6))
ax.add_patch(Polygon([(3.2, 2.4), (4.6, 2.4), (4.9, 3.6), (3.8, 4.2), (2.9, 3.4)],
                     facecolor="#C44E52", alpha=0.6))
ax.set(title="shapes (matplotlib)", xlabel="x", ylabel="y",
       xlim=(0, 6), ylim=(0, 5))
plt.show()

### fill / arrow
`fill` closes a polygon from x/y sequences; `arrow` draws a single vector.

In [ ]:
fa = [i * 0.2 for i in range(32)]
fy = [math.sin(t) for t in fa]
fig, ax = pp.subplots()
ax.fill(fa, fy, alpha=0.5)
ax.arrow(1.0, -0.6, 2.5, 0.4)
ax.set(title="fill / arrow", xlabel="t", ylabel="amplitude")
fig

In [ ]:
# matplotlib reference (reuses fa, fy)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.fill(fa, fy, alpha=0.5)
ax.arrow(1.0, -0.6, 2.5, 0.4, head_width=0.06, length_includes_head=True)
ax.set(title="fill / arrow (matplotlib)", xlabel="t", ylabel="amplitude")
plt.show()

### text / annotate
Free text at data coordinates, and a labeled callout with a leader.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs])
ax.text(1.0, 0.8, "free text", weight="bold")
ax.annotate("first peak", (math.pi / 2, 1.0), xytext=(4.0, 0.55))
ax.set(title="text / annotate", xlabel="t", ylabel="amplitude")
fig

In [ ]:
# matplotlib reference (reuses xs)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs])
ax.text(1.0, 0.8, "free text", weight="bold")
ax.annotate("first peak", (math.pi / 2, 1.0), xytext=(4.0, 0.55),
            arrowprops=dict(arrowstyle="->"))
ax.set(title="text / annotate (matplotlib)", xlabel="t", ylabel="amplitude")
plt.show()

### twinx / inset_axes
Axes composition: a second y scale on the right, and an inset panel.

In [ ]:
fig, ax = pp.subplots(figsize=(420, 260))
ax.line(xs, [math.sin(x) for x in xs], label="sin")
ax.set(title="twinx / inset_axes", xlabel="t (s)", ylabel="amplitude")
tw = ax.twinx()
tw.line(xs, [math.exp(0.2 * x) for x in xs], color="#DD8452", label="growth")
tw.set(ylabel="growth")
ins = ax.inset_axes((0.62, 0.08, 0.34, 0.32))
ins.line(xs[:12], [math.sin(x) for x in xs[:12]])
fig

In [ ]:
# matplotlib reference (reuses xs)
fig, ax = plt.subplots(figsize=pt2in(420, 260))
ax.plot(xs, [math.sin(x) for x in xs], label="sin")
ax.set(title="twinx / inset_axes (mpl)", xlabel="t (s)", ylabel="amplitude")
tw = ax.twinx()
tw.plot(xs, [math.exp(0.2 * x) for x in xs], color="#DD8452", label="growth")
tw.set(ylabel="growth")
ins = ax.inset_axes((0.62, 0.08, 0.34, 0.32))
ins.plot(xs[:12], [math.sin(x) for x in xs[:12]])
plt.show()

### secondary_xaxis / secondary_yaxis
A derived scale on the opposite spine, tied to the primary by a
forward/inverse function pair.

Both were incomplete until the layout fix: nothing reserved space for a
secondary axis, so it drew at the primary's own plot edge - on `top` the
ticks landed under the title, and on `right` they were emitted past the
canvas edge and never appeared at all (which read as "draws a bare
spine"). `label=` was accepted and then never rendered, on any location.

All four locations now reserve their own band and draw their label, so
the cells below should match the matplotlib reference.


In [ ]:
fig, (a1, a2) = pp.subplots(1, 2, figsize=(560, 220))
a1.line(xs, [math.sin(x) for x in xs])
a1.set(title="secondary_xaxis", xlabel="t (s)", ylabel="amplitude")
a1.secondary_xaxis("top", functions=(lambda t: t * 1000, lambda t: t / 1000),
                   label="t (ms)")
a2.line(xs, [math.sin(x) for x in xs])
a2.set(title="secondary_yaxis", xlabel="t (s)", ylabel="amplitude")
a2.secondary_yaxis("right", functions=(lambda v: v * 100, lambda v: v / 100),
                   label="percent")
fig

In [ ]:
# matplotlib reference (reuses xs)
fig, (a1, a2) = plt.subplots(1, 2, figsize=pt2in(560, 220))
a1.plot(xs, [math.sin(x) for x in xs])
a1.set(title="secondary_xaxis (mpl)", xlabel="t (s)", ylabel="amplitude")
s1 = a1.secondary_xaxis("top",
                        functions=(lambda t: t * 1000, lambda t: t / 1000))
s1.set_xlabel("t (ms)")
a2.plot(xs, [math.sin(x) for x in xs])
a2.set(title="secondary_yaxis (mpl)", xlabel="t (s)", ylabel="amplitude")
s2 = a2.secondary_yaxis("right",
                        functions=(lambda v: v * 100, lambda v: v / 100))
s2.set_ylabel("percent")
fig.tight_layout()
plt.show()

### Polar scatter
The polar projection also takes `scatter` (section 8 covered `plot`).

In [ ]:
random.seed(3)
pth = [random.uniform(0, 2 * math.pi) for _ in range(160)]
pr = [random.betavariate(2, 3) for _ in range(160)]
fig, ax = pp.subplots(projection="polar")
ax.scatter(pth, pr, size=18, alpha=0.75)
ax.set(title="Polar scatter")
fig

In [ ]:
# matplotlib reference (reuses pth, pr)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE),
                       subplot_kw=dict(projection="polar"))
ax.scatter(pth, pr, s=18, alpha=0.75)
ax.set(title="Polar scatter (matplotlib)")
plt.show()

### 3D scatter / line
`projection="3d"` gives an `Axes3D`, with `scatter` and `plot`.

In [ ]:
random.seed(4)
n3 = 150
x3 = [random.gauss(0, 1) for _ in range(n3)]
y3 = [random.gauss(0, 1) for _ in range(n3)]
z3 = [random.gauss(0, 1) for _ in range(n3)]
ht = [i * 0.15 for i in range(120)]
hx = [math.cos(t) * (1 + 0.1 * t) for t in ht]
hy = [math.sin(t) * (1 + 0.1 * t) for t in ht]

fig, (a1, a2) = pp.subplots(1, 2, figsize=(560, 240), projection="3d")
a1.scatter(x3, y3, z3, size=14)
a1.set(title="3D scatter", xlabel="x", ylabel="y", zlabel="z")
a2.plot(hx, hy, ht)
a2.set(title="3D line", xlabel="x", ylabel="y", zlabel="t")
fig

In [ ]:
# matplotlib reference (reuses x3, y3, z3, hx, hy, ht)
fig, (a1, a2) = plt.subplots(1, 2, figsize=pt2in(560, 240),
                             subplot_kw=dict(projection="3d"))
a1.scatter(x3, y3, z3, s=14)
a1.set(title="3D scatter (mpl)", xlabel="x", ylabel="y", zlabel="z")
a2.plot(hx, hy, ht)
a2.set(title="3D line (mpl)", xlabel="x", ylabel="y", zlabel="t")
plt.show()

### surface / plot_wireframe
The shared field `Z` as a shaded surface and as a wireframe. pyplotrs takes
the 1D coordinate vectors directly; matplotlib wants the meshed 2D arrays.

In [ ]:
fig, (a1, a2) = pp.subplots(1, 2, figsize=(560, 240), projection="3d")
a1.surface(gx, gy, Z, cmap="viridis")
a1.set(title="surface", xlabel="x", ylabel="y", zlabel="z")
a2.plot_wireframe(gx, gy, Z)
a2.set(title="plot_wireframe", xlabel="x", ylabel="y", zlabel="z")
fig

In [ ]:
# matplotlib reference (reuses gx, gy, Z; meshes them as mpl requires)
import numpy as np

GX = np.array([[x for x in gx] for _ in gy])
GY = np.array([[y for _ in gx] for y in gy])
GZ = np.array(Z)
fig, (a1, a2) = plt.subplots(1, 2, figsize=pt2in(560, 240),
                             subplot_kw=dict(projection="3d"))
a1.plot_surface(GX, GY, GZ, cmap="viridis")
a1.set(title="surface (mpl)", xlabel="x", ylabel="y", zlabel="z")
a2.plot_wireframe(GX, GY, GZ)
a2.set(title="plot_wireframe (mpl)", xlabel="x", ylabel="y", zlabel="z")
plt.show()

### contour3d
Contour lines of `Z` drawn in the 3D box.

In [ ]:
fig, ax = pp.subplots(figsize=(360, 300), projection="3d")
ax.contour3d(gx, gy, Z, levels=12)
ax.set(title="contour3d", xlabel="x", ylabel="y", zlabel="z")
fig

In [ ]:
# matplotlib reference (reuses GX, GY, GZ from the cell above)
fig = plt.figure(figsize=pt2in(360, 300))
ax = fig.add_subplot(projection="3d")
ax.contour(GX, GY, GZ, levels=12)
ax.set(title="contour3d (mpl)", xlabel="x", ylabel="y", zlabel="z")
plt.show()

### bar3d
3D bars on a small grid of categories.

In [ ]:
bxs, bys, bzs, bdz = [], [], [], []
random.seed(5)
for i in range(5):
    for j in range(5):
        bxs.append(i)
        bys.append(j)
        bzs.append(0)
        bdz.append(1 + 4 * random.random())
fig, ax = pp.subplots(figsize=(360, 300), projection="3d")
ax.bar3d(bxs, bys, bzs, 0.7, 0.7, bdz)
ax.set(title="bar3d", xlabel="i", ylabel="j", zlabel="value")
fig

In [ ]:
# matplotlib reference (reuses bxs, bys, bzs, bdz)
fig = plt.figure(figsize=pt2in(360, 300))
ax = fig.add_subplot(projection="3d")
ax.bar3d(bxs, bys, bzs, 0.7, 0.7, bdz)
ax.set(title="bar3d (mpl)", xlabel="i", ylabel="j", zlabel="value")
plt.show()

### plot_trisurf
A surface over scattered (unstructured) points, triangulated.

In [ ]:
random.seed(6)
tpx = [random.uniform(-3, 3) for _ in range(120)]
tpy = [random.uniform(-3, 3) for _ in range(120)]
tpz = [math.sin(math.hypot(a, b)) for a, b in zip(tpx, tpy)]
fig, ax = pp.subplots(figsize=(360, 300), projection="3d")
ax.plot_trisurf(tpx, tpy, tpz, cmap="viridis")
ax.set(title="plot_trisurf", xlabel="x", ylabel="y", zlabel="z")
fig

In [ ]:
# matplotlib reference (reuses tpx, tpy, tpz)
fig = plt.figure(figsize=pt2in(360, 300))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(tpx, tpy, tpz, cmap="viridis")
ax.set(title="plot_trisurf (mpl)", xlabel="x", ylabel="y", zlabel="z")
plt.show()

### quiver3d / voxels
3D arrows, and a filled voxel volume.

In [ ]:
qx, qy, qz, qu, qv, qw = [], [], [], [], [], []
for i in range(3):
    for j in range(3):
        for k in range(3):
            qx.append(i); qy.append(j); qz.append(k)
            qu.append(-(j - 1)); qv.append(i - 1); qw.append(0.5)
vox = [[[(i + j + k) % 3 == 0 for k in range(6)] for j in range(6)]
       for i in range(6)]

fig, (a1, a2) = pp.subplots(1, 2, figsize=(560, 240), projection="3d")
a1.quiver3d(qx, qy, qz, qu, qv, qw, length=0.4)
a1.set(title="quiver3d", xlabel="x", ylabel="y", zlabel="z")
a2.voxels(vox, alpha=0.9)
a2.set(title="voxels", xlabel="x", ylabel="y", zlabel="z")
fig

In [ ]:
# matplotlib reference (reuses qx..qw, vox)
fig = plt.figure(figsize=pt2in(560, 240))
a1 = fig.add_subplot(1, 2, 1, projection="3d")
a2 = fig.add_subplot(1, 2, 2, projection="3d")
a1.quiver(qx, qy, qz, qu, qv, qw, length=0.4)
a1.set(title="quiver3d (mpl)", xlabel="x", ylabel="y", zlabel="z")
a2.voxels(np.array(vox), alpha=0.9)
a2.set(title="voxels (mpl)", xlabel="x", ylabel="y", zlabel="z")
plt.show()